# Lab3: 点提示驱动的交互式分割
## Point-Prompt Driven Interactive Segmentation

---

### 实验背景

点提示（Point Prompt）是 SAM 系列最基础也最灵活的提示方式。
用户只需在目标区域点击一下，模型就能理解要分割什么。

SAM 3 支持以下几种点提示策略：

| 方式 | 说明 | 适用场景 |
|------|------|---------|
| **单正例点** | 在目标上点一下 | 简单目标，语义明确 |
| **多正例点** | 在目标不同位置点多次 | 大目标、复杂形状 |
| **正+负例点** | 正点击目标，负点击背景误检区域 | 精确修正，排除干扰 |
| **点+文本** | 点击标注位置 + 文本描述语义 | 同时利用位置和语义信息 |

### 点提示 API

SAM 3 的 `Sam3Processor` 没有直接暴露点提示方法，需手动调用底层 API：

```python
points = torch.tensor([[[x, y]]], device=device)   # (n_pts, 1, 2) 归一化坐标
labels = torch.tensor([[1]], device=device)          # (n_pts, 1)   1=正例, 0=负例
state["geometric_prompt"].append_points(points, labels)
state = processor._forward_grounding(state)
```

In [ ]:
# ========== 环境准备 ==========
import os
import sys
import csv
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from datetime import datetime

from IPython.display import display, clear_output
import ipywidgets as widgets

# ===== 修复 matplotlib 中文显示 =====
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'SimSun', 'FangSong', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
print(f"Font: {plt.rcParams['font.sans-serif'][0]}")

%matplotlib inline
print("Environment Ready \u2705")

In [ ]:
# ========== 路径配置 ==========
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
CKPT_PATH = os.path.join(REPO_ROOT, "sam3.pt")
INPUT_DIR = os.path.join(REPO_ROOT, "Inputs", "RawImages")
CSV_PATH = os.path.join(INPUT_DIR, "试标注图像清单.csv")

LAB3_OUTPUT = os.path.join(REPO_ROOT, "Outputs", "Lab3_output")
os.makedirs(LAB3_OUTPUT, exist_ok=True)

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"Checkpoint not found: {CKPT_PATH}")

# 读取 CSV
image_notes = {}
if os.path.exists(CSV_PATH):
    with open(CSV_PATH, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            image_notes[row["编号"]] = row

# 获取图片列表
image_files = sorted([
    f for f in os.listdir(INPUT_DIR) if f.endswith("_RawImage.png")
])
print(f"Images: {len(image_files)}")
print(f"Output: {LAB3_OUTPUT}")

In [ ]:
# ========== 加载 SAM 3 模型 ==========
print("[⏳] Loading SAM 3...")
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"    Device: {device}")

model = build_sam3_image_model(
    checkpoint_path=CKPT_PATH,
    load_from_HF=False,
    device=device,
    eval_mode=True,
)
processor = Sam3Processor(model, device=device)
print(f"[\u2705] SAM 3 loaded ({sum(p.numel() for p in model.parameters())/1e6:.0f}M)")

### 操作说明

运行下方单元格后，通过控件设置点提示：

1. **选择图片** → 下拉菜单
2. **设置正例点** → 滑块控制 x, y 坐标（归一化 0~1）
3. **设置负例点（可选）** → 勾选启用，用滑块调位置
4. **文本提示（可选）** → 输入框，留空则纯点提示
5. **点击运行** → 执行推理

> \u2728 **使用技巧**：
> - 正例点放在目标**中心附近**效果最好
> - 如果结果包含多余区域，加一个负例点指向误检区域
> - 点+文本组合：用文本描述语义，用点指示精确位置

In [ ]:
# ========== 交互式点提示 + SAM 3 推理 ==========

_current_img = None
_current_img_id = None
_current_img_w = 800
_current_img_h = 600


def add_point_prompt(state, points_norm, labels_val):
    """添加点提示到 SAM 3 推理状态"""
    pts_tensor = torch.tensor(points_norm, device=device, dtype=torch.float32).view(-1, 1, 2)
    lbl_tensor = torch.tensor(labels_val, device=device, dtype=torch.long).view(-1, 1)
    state["geometric_prompt"].append_points(pts_tensor, lbl_tensor)
    return state


def update_image_display(img_id):
    global _current_img, _current_img_id, _current_img_w, _current_img_h
    filename = f"{img_id}_RawImage.png"
    img_path = os.path.join(INPUT_DIR, filename)
    if os.path.exists(img_path):
        _current_img = Image.open(img_path).convert("RGB")
        _current_img_id = img_id
        _current_img_w, _current_img_h = _current_img.size
        notes = image_notes.get(img_id, {})
        desc_output.clear_output()
        with desc_output:
            print(f"\ud83d\udcf7 {filename} ({_current_img_w}\u00d7{_current_img_h})")
            print(f"Notes: {notes.get('\u573a\u666f\u7279\u70b9', '')}")
        update_preview(None)


def update_preview(change):
    """刷新预览图，显示点和框"""
    if _current_img is None:
        return
    
    px = slider_px.value
    py = slider_py.value
    use_neg = neg_toggle.value
    nx = slider_nx.value
    ny = slider_ny.value
    
    preview_output.clear_output(wait=True)
    with preview_output:
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(np.array(_current_img))
        
        # 正例点（绿色圆点）
        ax.plot(px * _current_img_w, py * _current_img_h, 'o',
                markersize=15, markerfacecolor='lime', markeredgecolor='green',
                markeredgewidth=2, label=f'Positive ({px:.2f}, {py:.2f})')
        
        # 负例点（红色叉号）
        if use_neg:
            ax.plot(nx * _current_img_w, ny * _current_img_h, 'x',
                    markersize=18, markeredgecolor='red', markeredgewidth=3,
                    label=f'Negative ({nx:.2f}, {ny:.2f})')
        
        ax.legend(fontsize=9)
        ax.axis("off")
        plt.tight_layout()
        plt.show()


def run_point_seg(btn):
    """执行点提示分割"""
    if _current_img is None:
        print("[\u26a0\ufe0f] Select an image first")
        return
    
    text_prompt = text_input.value.strip()
    px = slider_px.value
    py = slider_py.value
    use_neg = neg_toggle.value
    nx = slider_nx.value
    ny = slider_ny.value
    
    # 构建点列表
    points = [[px, py]]
    labels = [1]  # 1 = positive
    if use_neg:
        points.append([nx, ny])
        labels.append(0)  # 0 = negative
    
    result_output.clear_output(wait=True)
    with result_output:
        print(f"[\ud83c\udfaf] Points: {points}")
        print(f"[\ud83c\udf89] Labels: {labels}")
        print(f"[\ud83d\udcdd] Text: '{text_prompt}'" if text_prompt else "[\ud83d\udcdd] Text: none (point-only)")
        
        try:
            state = processor.set_image(_current_img)
            
            if text_prompt:
                state = processor.set_text_prompt(prompt=text_prompt, state=state)
            else:
                # 纯点提示需要用 "visual" 占位
                text_outputs = model.backbone.forward_text(["visual"], device=device)
                state["backbone_out"].update(text_outputs)
                state["geometric_prompt"] = model._get_dummy_prompt()
            
            # 添加点提示
            state = add_point_prompt(state, points, labels)
            state = processor._forward_grounding(state)
            
            masks = state["masks"]
            scores = state["scores"]
            
            print(f"[\u2705] Found {len(masks)} instances\n")
            for i in range(len(masks)):
                area_pct = masks[i].sum().item() / (_current_img_w * _current_img_h) * 100
                print(f"    #{i}: score={scores[i].item():.3f}, area={area_pct:.1f}%")
            
            # ---- 可视化 ----
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))
            
            # 左：原图+点
            axes[0].imshow(np.array(_current_img))
            # 正例点
            for pt, lb in zip(points, labels):
                marker = 'o' if lb == 1 else 'x'
                color = 'lime' if lb == 1 else 'red'
                size = 15 if lb == 1 else 18
                axes[0].plot(pt[0] * _current_img_w, pt[1] * _current_img_h,
                            marker, markersize=size, markerfacecolor=color,
                            markeredgecolor='darkgreen' if lb == 1 else 'darkred',
                            markeredgewidth=2)
            axes[0].set_title("Original + Points")
            axes[0].axis("off")
            
            # 右：mask 叠加
            overlay = np.zeros((_current_img_h, _current_img_w, 4), dtype=np.float32)
            cmap = plt.cm.tab10
            for i in range(len(masks)):
                m = masks[i].squeeze().cpu().numpy()
                overlay[m] = cmap(i % 10)[:3] + (0.5,)
                ys, xs = np.where(m)
                if len(xs) > 0:
                    cx_m, cy_m = xs[len(xs)//2], ys[len(ys)//2]
                    axes[1].text(cx_m, cy_m, f"{scores[i].item():.2f}",
                        color='white', fontsize=9, ha='center', va='center',
                        bbox=dict(boxstyle='round', facecolor='black', alpha=0.6))
            
            axes[1].imshow(np.array(_current_img))
            axes[1].imshow(overlay)
            mode = f"'{text_prompt}' + " if text_prompt else ""
            neg_str = " + neg" if use_neg else ""
            axes[1].set_title(f"{mode}{len(points)}pt{neg_str} | {len(masks)} instances")
            axes[1].axis("off")
            
            plt.tight_layout()
            plt.show()
            
            # ---- 保存 ----
            ts = datetime.now().strftime("%H%M%S")
            mode_tag = "point" if not text_prompt else "text+point"
            save_dir = os.path.join(LAB3_OUTPUT, f"{_current_img_id}_{mode_tag}_{ts}")
            os.makedirs(save_dir, exist_ok=True)
            fig.savefig(os.path.join(save_dir, "result.png"), dpi=150, bbox_inches="tight")
            for i in range(len(masks)):
                mask_np = masks[i].squeeze().cpu().numpy().astype(np.uint8) * 255
                Image.fromarray(mask_np).save(
                    os.path.join(save_dir, f"mask_{i:02d}_{scores[i].item():.3f}.png"))
            print(f"\n[\ud83d\udcbe] Saved: {save_dir}")
            
        except Exception as e:
            import traceback
            traceback.print_exc()
            print(f"[\u274c] Error: {e}")


# ========== 创建 UI ==========

# 图片选择
img_selector = widgets.Dropdown(
    options=[(f.replace('_RawImage.png',''), f.replace('_RawImage.png','')) for f in image_files],
    description='Image:', layout={'width': '180px'}
)

# 文本提示（可选）
text_input = widgets.Text(
    value='',
    description='Text (opt):',
    placeholder='Leave empty for point-only',
    layout={'width': '400px'}
)

# 正例点坐标
slider_px = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01,
                                description='Pos X:', layout={'width': '450px'},
                                readout_format='.2f')
slider_py = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01,
                                description='Pos Y:', layout={'width': '450px'},
                                readout_format='.2f')

# 负例点开关 + 坐标
neg_toggle = widgets.Checkbox(value=False, description='\u2796 Add negative point',
                              layout={'width': '200px'})
slider_nx = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.01,
                                description='Neg X:', layout={'width': '450px'},
                                readout_format='.2f', disabled=True)
slider_ny = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.01,
                                description='Neg Y:', layout={'width': '450px'},
                                readout_format='.2f', disabled=True)

def toggle_neg(change):
    slider_nx.disabled = not change.new
    slider_ny.disabled = not change.new
neg_toggle.observe(toggle_neg, names='value')

# 运行按钮
run_btn = widgets.Button(
    description='\ud83c\udfaf Run Point Segmentation',
    button_style='success',
    layout={'width': '280px', 'height': '40px'}
)
run_btn.on_click(run_point_seg)

# 输出区域
desc_output = widgets.Output()
preview_output = widgets.Output()
result_output = widgets.Output()

# ---- 布局 ----
controls = widgets.VBox([
    widgets.HBox([img_selector, text_input]),
    widgets.HTML("<b>\ud83d\udfe2 Positive point (green circle):</b>"),
    widgets.HBox([slider_px, slider_py]),
    widgets.HBox([neg_toggle]),
    widgets.HBox([slider_nx, slider_ny]),
    run_btn,
    desc_output,
])

# ---- 绑定事件 ----
img_selector.observe(lambda c: update_image_display(c.new) if c.new else None, names='value')
for s in [slider_px, slider_py, slider_nx, slider_ny]:
    s.observe(update_preview, names='value')
neg_toggle.observe(update_preview, names='value')

# ---- 显示 ----
display(controls)
display(widgets.HTML("<b>Preview:</b>"))
display(preview_output)
display(widgets.HTML("<b>Result:</b>"))
display(result_output)

# 加载第一张图
if image_files:
    first_id = image_files[0].replace('_RawImage.png', '')
    img_selector.value = first_id

### 四种提示方式对比

运行下方单元格，用**同样的目标区域**对比四种提示方式的效果。

这种方式是评估最适合你们任务的提示策略。

In [ ]:
# ========== 四种提示方式对比实验 ==========
if _current_img is None:
    print("[\u26a0\ufe0f] Run the cell above first to select an image")
else:
    print("=" * 70)
    print("Four-Way Comparison: same target, different prompts")
    print("=" * 70)
    
    px = slider_px.value
    py = slider_py.value
    use_neg = neg_toggle.value
    nx = slider_nx.value
    ny = slider_ny.value
    text_prompt = text_input.value.strip() or "swallowtail ridge"
    
    # 准备坐标
    points_pos = [[px, py]]
    labels_pos = [1]
    if use_neg:
        points_pos.append([nx, ny])
        labels_pos.append(0)
    
    # 用于框提示的归一化框（以正例点为中心的小框）
    box_size = 0.2
    norm_box = [px, py, box_size, box_size]
    
    # ---- A: 纯点提示 ----
    print("\n[A] Point-only prompt")
    state = processor.set_image(_current_img)
    txt_out = model.backbone.forward_text(["visual"], device=device)
    state["backbone_out"].update(txt_out)
    state["geometric_prompt"] = model._get_dummy_prompt()
    state = add_point_prompt(state, points_pos, labels_pos)
    state = processor._forward_grounding(state)
    masks_a, scores_a = state["masks"], state["scores"]
    print(f"    Found {len(masks_a)} instances")
    
    # ---- B: 文本+点 ----
    print(f"[B] Text + Point (text='{text_prompt}')")
    state = processor.set_image(_current_img)
    state = processor.set_text_prompt(prompt=text_prompt, state=state)
    state = add_point_prompt(state, points_pos, labels_pos)
    state = processor._forward_grounding(state)
    masks_b, scores_b = state["masks"], state["scores"]
    print(f"    Found {len(masks_b)} instances")
    
    # ---- C: 文本+框 ----
    print(f"[C] Text + Box (text='{text_prompt}')")
    state = processor.set_image(_current_img)
    state = processor.set_text_prompt(prompt=text_prompt, state=state)
    state = processor.add_geometric_prompt(box=norm_box, label=True, state=state)
    masks_c, scores_c = state["masks"], state["scores"]
    print(f"    Found {len(masks_c)} instances")
    
    # ---- D: 纯文本 ----
    print(f"[D] Text-only (text='{text_prompt}')")
    state = processor.set_image(_current_img)
    state = processor.set_text_prompt(prompt=text_prompt, state=state)
    masks_d, scores_d = state["masks"], state["scores"]
    print(f"    Found {len(masks_d)} instances")
    
    # ---- 可视化 ----
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    titles = [
        f"[A] Point-only ({len(masks_a)})",
        f"[B] Text+Point ({len(masks_b)})",
        f"[C] Text+Box ({len(masks_c)})",
        f"[D] Text-only ({len(masks_d)})",
    ]
    all_masks = [masks_a, masks_b, masks_c, masks_d]
    
    for idx, (ax, masks, title) in enumerate(zip(axes.flatten(), all_masks, titles)):
        overlay = np.zeros((_current_img_h, _current_img_w, 4), dtype=np.float32)
        colors = plt.cm.tab10(np.linspace(0, 1, max(len(masks), 1)))
        for i in range(len(masks)):
            m = masks[i].squeeze().cpu().numpy()
            overlay[m] = colors[i % len(colors)][:3] + (0.5,)
        
        # 在 point-only 和 text+point 上画点
        ax.imshow(np.array(_current_img))
        if idx < 2:  # A, B have point markers
            for pt, lb in zip(points_pos, labels_pos):
                marker = 'o' if lb == 1 else 'x'
                color = 'lime' if lb == 1 else 'red'
                s = 15 if lb == 1 else 18
                ax.plot(pt[0]*_current_img_w, pt[1]*_current_img_h, marker,
                        markersize=s, markerfacecolor=color,
                        markeredgecolor='green' if lb==1 else 'darkred', markeredgewidth=2)
        if idx == 2:  # C has box marker
            x1 = (px - box_size/2) * _current_img_w
            y1 = (py - box_size/2) * _current_img_h
            bw = box_size * _current_img_w
            bh = box_size * _current_img_h
            rect = plt.Rectangle((x1, y1), bw, bh, linewidth=2,
                                 edgecolor='lime', facecolor='none', linestyle='--')
            ax.add_patch(rect)
        ax.imshow(overlay)
        ax.set_title(title, fontsize=11)
        ax.axis("off")
    
    plt.tight_layout()
    ts = datetime.now().strftime("%H%M%S")
    save_path = os.path.join(LAB3_OUTPUT, f"{_current_img_id}_4way_{ts}.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\n[\ud83d\udcbe] Saved: {save_path}")

### 分析与记录

针对每次实验，可以记录以下指标：

| 维度 | 记录内容 |
|------|---------|
| **检出数** | 每种方式检出了几个实例 |
| **最高分** | 最高置信度分数 |
| **过分割** | 是否把非目标区域也分割了 |
| **欠分割** | 目标是否被完整覆盖 |
| **边界质量** | mask 边缘是否贴合目标轮廓 |
| **耗时** | 大概耗时（秒） |

### 提示策略指南

| 场景 | 推荐方式 | 理由 |
|------|---------|------|
| 目标语义明确（屋顶、墙） | **纯文本（Lab1）** | 批量快、全图搜索 |
| 目标位置已知 | **文本+框（Lab2）** | 减少假阳性 |
| 目标位置精确已知 | **文本+点（本实验）** | 精确定位 |
| 目标细长/形状不规则 | **多点提示** | 多点共同约束形状 |
| 需要排除相邻干扰 | **正+负例点** | 负例排除误检 |
| 先文本找到大致位置 | **文本→人工确认→加框/点修正** | 最佳工作流 |